In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from scipy.spatial import cKDTree

import sys
sys.path.append("../")
from src.california_data_preparation import merge_earthquake_and_stress
from src.california_data_preparation import exclude_geotherm
from src.california_data_preparation import Mc_estimation
from src.plot import plot_GoFcurve
from src.plot import plot_Mc_confirmation
from src.plot import plot_spatial_distribution
from src.plot import plot_spatial_distribution_GTF
from src.plot import plot_CMFD_dataset1GTF

In [2]:
output_dir_data = Path("../data/california/processed")
output_dir_data.mkdir(parents=True, exist_ok=True)

output_dir_fig = Path("../figures/dataset")
output_dir_fig.mkdir(parents=True, exist_ok=True)

In [3]:
# =========================================================
# Dataset1
# =========================================================
## earthquake catalog
with open("../data/california/raw/dataset1/focmec_ca_final.pickle", "rb") as f:
    data = pickle.load(f)
df1_earthquake = pd.DataFrame(data)
df1_earthquake = df1_earthquake[["year", "month", "day", "hour", "minute", "second", "lat_reloc", "lon_reloc", "dep_reloc", "mag", "strike", "dip", "rake"]]
df1_earthquake.columns = ["year_event", "month_event", "day_event", "hour_event", "minute_event", "second_event", "latitude_event", "longitude_event", "depth_event", "magnitude", "strike", "dip", "rake"]

## stress catalog (2D)
# components of the stress tensor are initially on the N-E-D coordinate
with open("../data/california/raw/dataset1/stress_CA_2D_N35_5km_mu_06.pickle", "rb") as f:
    data = pickle.load(f)
lon = data["lon"]
lat = data["lat"]
lon_grid, lat_grid = np.meshgrid(lon, lat, indexing="ij")
df1_stress = pd.DataFrame({
    "lon": lon_grid.flatten(),
    "lat": lat_grid.flatten()
})
for k, v in data.items():
    if isinstance(v, np.ndarray) and v.shape == (240, 220):
        df1_stress[k] = v.flatten()
df1_stress = df1_stress.dropna()
df1_stress = df1_stress[["lat", "lon", "s11", "s12", "s13", "s22", "s23", "s33", "R"]]
df1_stress.columns = ["latitude_stress", "longitude_stress", "Snn", "Sne", "Snd", "See", "Sed", "Sdd", "R"]

## merge earthquake and stress data
# use only events that have stress estimation within 10km from epicenters
dataset1 = merge_earthquake_and_stress(df1_earthquake, df1_stress, max_dist_km=10)

## Mc estimation (Goodness of Fit)
Mc_idx1, Mc_candidate1, GoF1, b1, CFMD_obs1, CFMD_pred1, magnitude_centers1 = Mc_estimation(dataset1["magnitude"], GoF_threshold=97.5)
fig, ax, Mc1 = plot_GoFcurve(Mc_idx1, Mc_candidate1, GoF1, GoF_threshold=97.5)
fig.savefig(output_dir_fig/"dataset1_GoF.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

fig, ax = plot_Mc_confirmation(Mc_idx1, CFMD_obs1, CFMD_pred1, b1, Mc1, magnitude_centers1)
fig.savefig(output_dir_fig/"dataset1_Mc_confirm.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## draw map of epicenters
dataset1_Mc = dataset1[dataset1["magnitude"]>=Mc1]
fig, ax = plot_spatial_distribution(
    dataset1_Mc["latitude_event"].to_numpy(),
    dataset1_Mc["longitude_event"].to_numpy(),
    color="#0072B2"
)
fig.savefig(output_dir_fig/"dataset1_Mc_eqmap.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## draw map of refered stress catalogs
fig, ax = plot_spatial_distribution(
    dataset1_Mc["latitude_stress"].to_numpy(),
    dataset1_Mc["longitude_stress"].to_numpy(),
    color="red"
)
fig.savefig(output_dir_fig/"dataset1_stressmap.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## save as a csv file
dataset1_Mc.to_csv(output_dir_data/"dataset1.csv", index=False)

C:\Users\agata\AppData\Local\Temp\ipykernel_33404\159184136.py:6: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)
C:\Users\agata\AppData\Local\Temp\ipykernel_33404\159184136.py:14: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functi

In [4]:
# =========================================================
# Dataset2
# =========================================================
## earthquake catalog
FOCAL_MECH_PATH = "../data/california/raw/dataset2/1981-2010_Focal_Mechanism_Catalog.hash"
columns_earthquake = [
    "year_event", "month_event", "day_event", "hour_event", "minute_event", "second_event",
    "event_id", "latitude_event", "longitude_event", "depth_event",
    "magnitude", "strike", "dip", "rake",
    "fault_plane_uncertainty", "auxiliary_fault_plane_uncertainty", 
    "number_of_P_wave_first_motions", "misfit_of_first_motions",
    "number_of_sp_amplitude_ratios", "average_log_misfit", "quality"
]
df2_earthquake = pd.read_csv(FOCAL_MECH_PATH, sep=r'\s+', names=columns_earthquake, comment="#", engine="python", header=None)
df2_earthquake = df2_earthquake[["year_event", "month_event", "day_event", "hour_event", "minute_event", "second_event", "latitude_event", "longitude_event", "depth_event", "magnitude", "strike", "dip", "rake"]]

## stress catalog (2D)
# components of the stress tensor are initially on the E-N-U coordinate
STRESS_MODEL_PATH = "../data/california/raw/dataset2/YHSM-2013.csv"
columns_stress_raw = [
    "longitude_stress", "latitude_stress", "depth_stress", "See", "Sen", "Seu", "Snn", "Snu", "Suu", 
    "SHmax", "SHmax_unc", "phi", "R", "Aphi", "iso", "dif", "mss", 
    "S1", "S2", "S3", "V1x", "V1y", "V1z", "V2x", "V2y", "V2z", 
    "V3x", "V3y", "V3z", "V1pl", "V2pl", "V3pl", "V1azi", "V2azi", "V3azi"
]
df2_stress_raw = pd.read_csv(STRESS_MODEL_PATH, comment="#", names=columns_stress_raw, header=None)
df2_stress_raw = df2_stress_raw[["longitude_stress", "latitude_stress", "See", "Sen", "Seu", "Snn", "Snu", "Suu", "R"]]

Snn = df2_stress_raw["Snn"]
Sne = df2_stress_raw["Sen"]
Snd = - df2_stress_raw["Snu"]
See = df2_stress_raw["See"]
Sed = - df2_stress_raw["Seu"]
Sdd = df2_stress_raw["Suu"]
df2_stress = df2_stress_raw.copy()
df2_stress.columns = ["longitude_stress", "latitude_stress", "Snn", "Sne", "Snd", "See", "Sed", "Sdd", "R"]
df2_stress["Snn"] = Snn
df2_stress["Sne"] = Sne
df2_stress["Snd"] = Snd
df2_stress["See"] = See
df2_stress["Sed"] = Sed
df2_stress["Sdd"] = Sdd
df2_stress = df2_stress[["latitude_stress", "longitude_stress", "Snn", "Sne", "Snd", "See", "Sed", "Sdd", "R"]]

## merge earthquake and stress data
# use only events that have stress estimation within 10km from hypocenters
dataset2 = merge_earthquake_and_stress(df2_earthquake, df2_stress, max_dist_km=10)

## Mc estimation (Goodness of Fit)
Mc_idx2, Mc_candidate2, GoF2, b2, CFMD_obs2, CFMD_pred2, magnitude_centers2 = Mc_estimation(dataset2["magnitude"], GoF_threshold=97.5)
fig, ax, Mc2 = plot_GoFcurve(Mc_idx2, Mc_candidate2, GoF2, GoF_threshold=97.5)
fig.savefig(output_dir_fig/"dataset2_GoF.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

fig, ax = plot_Mc_confirmation(Mc_idx2, CFMD_obs2, CFMD_pred2, b2, Mc2, magnitude_centers2)
fig.savefig(output_dir_fig/"dataset2_Mc_confirm.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## drow map of epicenters
dataset2_Mc = dataset2[dataset2["magnitude"]>=Mc2]
fig, ax = plot_spatial_distribution(
    dataset2_Mc["latitude_event"].to_numpy(),
    dataset2_Mc["longitude_event"].to_numpy(),
    color="#0072B2"
)
fig.savefig(output_dir_fig/"dataset2_Mc_eqmap.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## draw map of refered stress catalogs
fig, ax = plot_spatial_distribution(
    dataset2_Mc["latitude_stress"].to_numpy(),
    dataset2_Mc["longitude_stress"].to_numpy(),
    color="red"
)
fig.savefig(output_dir_fig/"dataset2_stressmap.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

## save as a csv file
dataset2_Mc.to_csv(output_dir_data/"dataset2.csv", index=False)

In [5]:
### Dataset1 with consideration of Geothermal fields (GTF) ###

# Distance from Geothermal fields [km]
r_km = 10
dataset1_Mc_GTF_in, dataset1_Mc_GTF_out = exclude_geotherm(dataset1_Mc, r_km)

## geothermal group ##
# draw map of epicenters
fig, ax = plot_spatial_distribution_GTF(
    dataset1_Mc_GTF_in["latitude_event"].to_numpy(),
    dataset1_Mc_GTF_in["longitude_event"].to_numpy(),
)
fig.savefig(output_dir_fig/f"dataset1_Mc_GTF_{r_km:.0f}in_map.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

# save as a csv file
dataset1_Mc_GTF_in.to_csv(output_dir_data/f"dataset1_GTF{r_km:.0f}in.csv", index=False)


## non-geothermal group ##
# draw map of epicenters
fig, ax = plot_spatial_distribution_GTF(
    dataset1_Mc_GTF_out["latitude_event"].to_numpy(),
    dataset1_Mc_GTF_out["longitude_event"].to_numpy(),
)
fig.savefig(output_dir_fig/f"dataset1_Mc_GTF_{r_km:.0f}out_map.svg", bbox_inches="tight", dpi=300)
plt.close(fig)

# save as a csv file
dataset1_Mc_GTF_out.to_csv(output_dir_data/f"dataset1_GTF{r_km:.0f}out.csv", index=False)


## draw magnitude-frequency distribution
magnitude_in = dataset1_Mc_GTF_in["magnitude"].to_numpy()
magnitude_out = dataset1_Mc_GTF_out["magnitude"].to_numpy()

bval_in = np.log10(np.e) / (magnitude_in.mean() - Mc1)
bval_out = np.log10(np.e) / (magnitude_out.mean() - Mc1)

fig, ax = plot_CMFD_dataset1GTF(magnitude_in, magnitude_out, bval_in, bval_out, Mc1)
fig.savefig(output_dir_fig/f"dataset1_Mc_GTF_{r_km:.0f}_CMFD.svg", bbox_inches="tight", dpi=300)
plt.close(fig)